# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
```
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json
```

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s. All entities are referenced by their `@id` field.

In [ ]:
# List all record sets in the dataset, with their @id and name
print("Available record sets:")
for record_set in dataset.record_sets:
    print(f"- record_set @id: {record_set.id}, name: {record_set.name}")

# For each recordset, print its fields and their @id
for record_set in dataset.record_sets:
    print(f"\nFields in record_set '{record_set.name}' (@id: {record_set.id}):")
    for field in record_set.fields:
        print(f"    - field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")

## 3. Data Extraction
For this section, we will load the data from the main record set into a DataFrame for analysis. All references to record set and field or column use their `@id`s.
To proceed, choose the primary record set of interest according to the printed overview.

In [ ]:
# Find the main record set (@id) - for this dataset, let's choose the one whose name contains 'protein', if present.
main_record_set_id = None
for record_set in dataset.record_sets:
    if 'protein' in record_set.name.lower():
        main_record_set_id = record_set.id
        break

# If not found, use the first record set
if main_record_set_id is None and dataset.record_sets:
    main_record_set_id = dataset.record_sets[0].id

if main_record_set_id is not None:
    print(f"Using record set: {main_record_set_id}")
else:
    raise ValueError("No record sets found in the dataset.")

# Optionally extract all record sets if more than one
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

print(f"Fields in DataFrame for {main_record_set_id}:\n", dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll apply data processing steps, including filtering by a numeric field, normalization, and (optionally) grouping by another field. All variable references use field `@id`s.

> **Note:** Please adjust the chosen field `@id`s to match those printed in the **Overview** section above. Below, we select the first numeric field available for analysis.

In [ ]:
# Automatically select a numeric field for EDA
df = dataframes[main_record_set_id]

# Try to pick the first column that is numeric
numeric_field_id = None
for col in df.columns:
    # Try to convert to numeric to test
    try:
        sample_nonnull = df[col].dropna()
        # Only treat as numeric if >50% of values can be parsed
        if sample_nonnull.size > 0:
            vals = pd.to_numeric(sample_nonnull, errors='coerce')
            if vals.notna().sum() / sample_nonnull.size > 0.5:
                numeric_field_id = col
                df[col] = vals  # Coerce to numeric
                break
    except Exception:
        pass

if numeric_field_id is None:
    raise ValueError('No numeric field found for EDA.')

print(f"Using numeric field: {numeric_field_id}")

# Filter (arbitrary threshold, e.g., values > 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records where {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to automatically pick a categorical/group field (not the numeric field)
group_field_id = None
for col in df.columns:
    if col == numeric_field_id:
        continue
    # Heuristic: column with <= 20 unique values and type object
    if df[col].dtype == object and df[col].nunique() <= 20:
        group_field_id = col
        break

if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the data distributions for key numeric and (if available) categorical fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If grouping field is available, show boxplot
if group_field_id:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f'{numeric_field_id} by {group_field_id}')
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR^2 dataset using the `mlcroissant` library. We identified available record sets and fields (referenced by their `@id`), extracted tabular data, performed basic EDA operations—including filtering, normalization, and grouping—and visualized the data.

This workflow provides a strong foundation for deeper investigation or integration of this Omics dataset into downstream data science applications.

**Tip:** For more advanced analyses, consult the [mlcroissant documentation](https://github.com/mlcommons/croissant/tree/main/python/mlcroissant) and further tailor field IDs and transformations to your specific use case.